In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train_identity=pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction=pd.read_csv('ieee-fraud-detection/train_transaction.csv')

In [3]:
# Columns we initially care about
transaction_cols = [
    "TransactionID",
    "isFraud",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",

    # Payment/card information
    "card1", "card2", "card3", "card4", "card5", "card6",

    # Address information
    "addr1", "addr2",

    # Email information
    "P_emaildomain",
    "R_emaildomain",
]

identity_cols = [
    "TransactionID",
    "DeviceType",
    "DeviceInfo",
] + [f"id_{i:02d}" for i in range(1, 39)]

tx = train_transaction[transaction_cols].copy()

identity = train_identity[identity_cols].copy()

df = tx.merge(
    identity,
    on="TransactionID",
    how="left"
)

In [4]:
import pandas as pd
import networkx as nx
from collections import Counter

# ============================================================
# G1: DEVICE-ONLY ENTITY RESOLUTION GRAPH
# ============================================================

GRAPH_FEATURES = ["DeviceInfo"]

# Make sure card1 is treated as the card identity
work = df[["card1", "isFraud"] + GRAPH_FEATURES].copy()

# Remove rows without a card
work = work.dropna(subset=["card1"])

# Clean entity values
for feature in GRAPH_FEATURES:
    work[feature] = work[feature].astype("string")
    work[feature] = work[feature].str.strip()

# Do not create edges for missing entities
work = work.dropna(subset=GRAPH_FEATURES)

print("Transactions used:", len(work))
print("Unique cards:", work["card1"].nunique())

# ------------------------------------------------------------
# CARD NODES
# ------------------------------------------------------------

card_stats = (
    work.groupby("card1")
    .agg(
        transactions=("card1", "size"),
        frauds=("isFraud", "sum"),
    )
)

card_stats["fraud_rate"] = (
    card_stats["frauds"] / card_stats["transactions"]
)

# ------------------------------------------------------------
# ENTITY → CARDS
# ------------------------------------------------------------

entity_cards = (
    work.groupby("DeviceInfo")["card1"]
    .agg(lambda x: set(x))
)

print("\nUnique DeviceInfo entities:", len(entity_cards))

# ------------------------------------------------------------
# BUILD GRAPH
# ------------------------------------------------------------

G = nx.Graph()

# Add card nodes
for card, row in card_stats.iterrows():

    G.add_node(
        f"card_{card}",
        node_type="card",
        card_id=card,
        transactions=int(row["transactions"]),
        frauds=int(row["frauds"]),
        fraud_rate=float(row["fraud_rate"])
    )

# Add entity nodes + edges
for device, cards in entity_cards.items():

    entity_node = f"device::{device}"

    G.add_node(
        entity_node,
        node_type="entity",
        feature="DeviceInfo",
        value=device,
        card_count=len(cards)
    )

    for card in cards:

        G.add_edge(
            f"card_{card}",
            entity_node,
            relation="shared_DeviceInfo"
        )

# ============================================================
# BASIC GRAPH STATISTICS
# ============================================================

print("\n================ GRAPH G1 ================")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

card_nodes = [
    n for n, d in G.nodes(data=True)
    if d["node_type"] == "card"
]

entity_nodes = [
    n for n, d in G.nodes(data=True)
    if d["node_type"] == "entity"
]

print("Card nodes:", len(card_nodes))
print("Entity nodes:", len(entity_nodes))

# Connected components
components = list(nx.connected_components(G))

print("Connected components:", len(components))

component_sizes = sorted(
    [len(c) for c in components],
    reverse=True
)

print("Largest components:", component_sizes[:20])

Transactions used: 118666
Unique cards: 7853

Unique DeviceInfo entities: 1786

================ GRAPH G1 ================
Nodes: 9639
Edges: 26141
Card nodes: 7853
Entity nodes: 1786
Connected components: 50
Largest components: [9533, 5, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [7]:
# ============================================================
# TOP ENTITY HUBS
# ============================================================

entity_hubs = []

for node, data in G.nodes(data=True):

    if data["node_type"] == "entity":

        entity_hubs.append({
            "entity": data["value"],
            "cards": data["card_count"]
        })

hub_df = (
    pd.DataFrame(entity_hubs)
    .sort_values("cards", ascending=False)
)

print("\n================ TOP DEVICE HUBS ================")

print(hub_df.head(15).to_string(index=False))


================ TOP DEVICE HUBS ================
               entity  cards
              Windows   4846
           iOS Device   3104
                MacOS   2287
          Trident/7.0   1868
              rv:11.0    702
              rv:57.0    422
SM-G930V Build/NRD90M    167
SM-G950U Build/NRD90M    164
SM-G955U Build/NRD90M    163
              SAMSUNG    157
              rv:59.0    157
              rv:58.0    146
              rv:52.0    140
SM-N950U Build/NMF26X    133
              rv:48.0    102


In [14]:
THRESHOLDS = [100, 250, 500, 1000, 2000, None]

results = []

for threshold in THRESHOLDS:

    G = nx.Graph()

    # Card nodes
    for card, row in card_stats.iterrows():
        G.add_node(
            f"card_{card}",
            node_type="card",
            card_id=card,
            transactions=int(row["transactions"]),
            frauds=int(row["frauds"]),
            fraud_rate=float(row["fraud_rate"])
        )

    kept_entities = 0

    # Entity nodes + edges
    for device, cards in entity_cards.items():

        n_cards = len(cards)

        if n_cards < 2:
            continue

        if threshold is not None and n_cards > threshold:
            continue

        kept_entities += 1

        entity_node = f"device::{device}"

        G.add_node(
            entity_node,
            node_type="entity",
            feature="DeviceInfo",
            value=device,
            card_count=n_cards
        )

        for card in cards:
            G.add_edge(
                f"card_{card}",
                entity_node,
                relation="shared_DeviceInfo"
            )

    components = sorted(
        [len(c) for c in nx.connected_components(G)],
        reverse=True
    )

    card_count = sum(
        d["node_type"] == "card"
        for _, d in G.nodes(data=True)
    )

    largest = components[0] if components else 0
    second = components[1] if len(components) > 1 else 0

    results.append({
        "thres": threshold if threshold is not None else "NONE",
        "ent_nodes": kept_entities,
        "tot_nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "comps": len(components),
        "largest_comp": largest,
        # "2nd_comp": second,
        "largest_comp_card%": round(
            100 * largest / card_count, 2
        )
    })

pd.DataFrame(results)

,thres,ent_nodes,tot_nodes,edges,comps,largest_comp,largest_comp_card%
0,100,1222,9075,11034,5726,3316,42.23
1,250,1231,9084,12363,5340,3719,47.36
2,500,1232,9085,12785,5193,3870,49.28
3,1000,1233,9086,13487,4898,4166,53.05
4,2000,1234,9087,15355,3955,5112,65.10
5,NONE,1237,9090,25592,50,9030,114.99


In [15]:
# ============================================================
# G1 FILTERED ER GRAPH
# ============================================================

MAX_CARDS_PER_ENTITY = 1000

G1 = nx.Graph()

# ------------------------------------------------------------
# Card nodes
# ------------------------------------------------------------

for card, row in card_stats.iterrows():

    G1.add_node(
        f"card_{card}",
        node_type="card",
        card_id=card,
        transactions=int(row["transactions"]),
        frauds=int(row["frauds"]),
        fraud_rate=float(row["fraud_rate"])
    )

# ------------------------------------------------------------
# Entity nodes
# ------------------------------------------------------------

kept_entities = 0
discarded_entities = 0

for device, cards in entity_cards.items():

    card_count = len(cards)

    if card_count < 2:
        continue

    if card_count > MAX_CARDS_PER_ENTITY:
        discarded_entities += 1
        continue

    kept_entities += 1

    entity_node = f"device::{device}"

    G1.add_node(
        entity_node,
        node_type="entity",
        feature="DeviceInfo",
        value=device,
        card_count=card_count
    )

    for card in cards:

        G1.add_edge(
            f"card_{card}",
            entity_node,
            relation="shared_DeviceInfo"
        )

print("\n================ FILTERED G1 ================")

print("Nodes:", G1.number_of_nodes())
print("Edges:", G1.number_of_edges())

print("Card nodes:",
      sum(d["node_type"] == "card"
          for _, d in G1.nodes(data=True)))

print("Entity nodes:",
      sum(d["node_type"] == "entity"
          for _, d in G1.nodes(data=True)))

print("Entities kept:", kept_entities)
print("Entities discarded:", discarded_entities)


================ FILTERED G1 ================
Nodes: 9086
Edges: 13487
Card nodes: 7853
Entity nodes: 1233
Entities kept: 1233
Entities discarded: 4


In [16]:
# ============================================================
# CARD-CARD PROJECTION
# ============================================================

card_graph = nx.Graph()

# Add cards
for card_node, data in G1.nodes(data=True):

    if data["node_type"] == "card":
        card_graph.add_node(
            card_node,
            **data
        )

# For every entity, connect all cards sharing it
for entity_node, entity_data in G1.nodes(data=True):

    if entity_data["node_type"] != "entity":
        continue

    cards = [
        n for n in G1.neighbors(entity_node)
        if G1.nodes[n]["node_type"] == "card"
    ]

    # connect cards sharing this entity
    for i in range(len(cards)):
        for j in range(i + 1, len(cards)):

            c1 = cards[i]
            c2 = cards[j]

            if card_graph.has_edge(c1, c2):

                card_graph[c1][c2]["shared_entities"] += 1

            else:

                card_graph.add_edge(
                    c1,
                    c2,
                    shared_entities=1
                )

print("\n================ CARD GRAPH ================")

print("Card nodes:", card_graph.number_of_nodes())
print("Card-card edges:", card_graph.number_of_edges())


================ CARD GRAPH ================
Card nodes: 7853
Card-card edges: 430841


In [17]:
# ============================================================
# CONNECTED CARD NETWORKS
# ============================================================

components = list(
    nx.connected_components(card_graph)
)

components = sorted(
    components,
    key=len,
    reverse=True
)

print("\n================ MAJOR NETWORKS ================")

for i, component in enumerate(components[:20], start=1):

    cards = list(component)

    total_transactions = sum(
        card_graph.nodes[c]["transactions"]
        for c in cards
    )

    total_frauds = sum(
        card_graph.nodes[c]["frauds"]
        for c in cards
    )

    fraud_rate = (
        total_frauds / total_transactions
        if total_transactions > 0
        else 0
    )

    print(
        f"Network {i}: "
        f"cards={len(cards)}, "
        f"transactions={total_transactions}, "
        f"frauds={total_frauds}, "
        f"fraud_rate={fraud_rate:.4f}"
    )


================ MAJOR NETWORKS ================
Network 1: cards=2944, transactions=105052, frauds=8038, fraud_rate=0.0765
Network 2: cards=3, transactions=7, frauds=0, fraud_rate=0.0000
Network 3: cards=2, transactions=5, frauds=0, fraud_rate=0.0000
Network 4: cards=2, transactions=7, frauds=0, fraud_rate=0.0000
Network 5: cards=2, transactions=4, frauds=0, fraud_rate=0.0000
Network 6: cards=2, transactions=7, frauds=0, fraud_rate=0.0000
Network 7: cards=2, transactions=4, frauds=0, fraud_rate=0.0000
Network 8: cards=2, transactions=6, frauds=0, fraud_rate=0.0000
Network 9: cards=2, transactions=7, frauds=0, fraud_rate=0.0000
Network 10: cards=2, transactions=5, frauds=0, fraud_rate=0.0000
Network 11: cards=2, transactions=8, frauds=0, fraud_rate=0.0000
Network 12: cards=2, transactions=3, frauds=0, fraud_rate=0.0000
Network 13: cards=1, transactions=3, frauds=0, fraud_rate=0.0000
Network 14: cards=1, transactions=1, frauds=0, fraud_rate=0.0000
Network 15: cards=1, transactions=2, f

In [18]:
import pandas as pd
import networkx as nx
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

CARD_COL = "card1"

ENTITY_FEATURES = [
    "DeviceInfo",
    "card2"
]

# Minimum number of distinct cards sharing an entity
MIN_CARDS_PER_ENTITY = 2


# ============================================================
# 1. BUILD BIPARTITE CARD -> ENTITY GRAPH
# ============================================================

G = nx.Graph()

# Add card nodes
cards = df[CARD_COL].dropna().unique()

for card in cards:
    G.add_node(
        f"card_{card}",
        node_type="card"
    )

entity_stats = {}

for feature in ENTITY_FEATURES:

    # Only rows having this feature
    temp = df[[CARD_COL, feature]].dropna().copy()

    # Important:
    # one card should connect only ONCE to an entity
    temp = temp.drop_duplicates([CARD_COL, feature])

    # How many distinct cards use each entity?
    card_counts = (
        temp.groupby(feature)[CARD_COL]
        .nunique()
    )

    # Keep entities shared by >= MIN_CARDS_PER_ENTITY cards
    valid_entities = card_counts[
        card_counts >= MIN_CARDS_PER_ENTITY
    ].index

    temp = temp[
        temp[feature].isin(valid_entities)
    ]

    # Add entity nodes + edges
    for value, group in temp.groupby(feature):

        entity_id = f"{feature}::{value}"

        G.add_node(
            entity_id,
            node_type="entity",
            feature=feature,
            value=value,
            card_count=len(group)
        )

        for card in group[CARD_COL].unique():

            G.add_edge(
                f"card_{card}",
                entity_id
            )

    entity_stats[feature] = len(valid_entities)

    print(
        f"{feature}: "
        f"{len(valid_entities)} entities kept"
    )


# ============================================================
# 2. BASIC GRAPH STATISTICS
# ============================================================

card_nodes = [
    n for n, d in G.nodes(data=True)
    if d["node_type"] == "card"
]

entity_nodes = [
    n for n, d in G.nodes(data=True)
    if d["node_type"] == "entity"
]

print("\n================ GRAPH G2 ================")

print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

print("Card nodes:", len(card_nodes))
print("Entity nodes:", len(entity_nodes))

print(
    "Connected components:",
    nx.number_connected_components(G)
)


# ============================================================
# 3. CARD-CARD PROJECTION
# ============================================================

card_graph = nx.Graph()

card_graph.add_nodes_from(card_nodes)

# entity -> cards
for entity in entity_nodes:

    connected_cards = [
        n for n in G.neighbors(entity)
        if G.nodes[n]["node_type"] == "card"
    ]

    # Every pair of cards sharing this entity
    for i in range(len(connected_cards)):

        for j in range(i + 1, len(connected_cards)):

            c1 = connected_cards[i]
            c2 = connected_cards[j]

            if card_graph.has_edge(c1, c2):

                card_graph[c1][c2]["shared_entities"] += 1

            else:

                card_graph.add_edge(
                    c1,
                    c2,
                    shared_entities=1
                )


print("\n================ CARD GRAPH ================")

print(
    "Card nodes:",
    card_graph.number_of_nodes()
)

print(
    "Card-card edges:",
    card_graph.number_of_edges()
)


# ============================================================
# 4. MAJOR CARD NETWORKS
# ============================================================

components = sorted(
    nx.connected_components(card_graph),
    key=len,
    reverse=True
)

print("\n================ MAJOR NETWORKS ================")

for i, component in enumerate(components[:20], 1):

    if len(component) < 2:
        continue

    subgraph = df[
        df[CARD_COL].astype(str).apply(
            lambda x: f"card_{x}" in component
        )
    ]

    transactions = len(subgraph)
    frauds = subgraph["isFraud"].sum()

    fraud_rate = (
        frauds / transactions
        if transactions > 0
        else 0
    )

    print(
        f"Network {i}: "
        f"cards={len(component)}, "
        f"transactions={transactions}, "
        f"frauds={frauds}, "
        f"fraud_rate={fraud_rate:.4f}"
    )

DeviceInfo: 1237 entities kept
card2: 443 entities kept

================ GRAPH G2 ================
Total nodes: 15233
Total edges: 39025
Card nodes: 13553
Entity nodes: 1680
Connected components: 209

================ CARD GRAPH ================
Card nodes: 13553
Card-card edges: 30442946

================ MAJOR NETWORKS ================
Network 1: cards=13329, transactions=587164, frauds=20646, fraud_rate=0.0352
Network 2: cards=10, transactions=39, frauds=1, fraud_rate=0.0256
Network 3: cards=3, transactions=172, frauds=0, fraud_rate=0.0000
Network 4: cards=3, transactions=121, frauds=0, fraud_rate=0.0000
Network 5: cards=2, transactions=76, frauds=0, fraud_rate=0.0000
Network 6: cards=2, transactions=60, frauds=0, fraud_rate=0.0000
Network 7: cards=2, transactions=76, frauds=1, fraud_rate=0.0132


In [20]:
# ============================================================
# STEP 6: FRAUD ANALYSIS OF G3 COMPONENTS
# ============================================================

# Map card -> transactions
card_stats = (
    df.groupby("card1")
      .agg(
          transactions=("TransactionID", "count"),
          frauds=("isFraud", "sum")
      )
)

card_stats["fraud_rate"] = (
    card_stats["frauds"] / card_stats["transactions"]
)


# Get connected components
components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

print("\n========== G3 FRAUD ANALYSIS ==========")

overall_fraud_rate = df["isFraud"].mean()

print(f"Overall fraud rate: {overall_fraud_rate:.4f}")

# Analyze top 20 components
results = []

for i, component in enumerate(components[:20], 1):

    cards = list(component)

    stats = card_stats.loc[
        card_stats.index.intersection(cards)
    ]

    transactions = stats["transactions"].sum()
    frauds = stats["frauds"].sum()

    if transactions == 0:
        continue

    fraud_rate = frauds / transactions

    results.append({
        "network": i,
        "cards": len(cards),
        "transactions": transactions,
        "frauds": frauds,
        "fraud_rate": fraud_rate,
        "lift": fraud_rate / overall_fraud_rate
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))


========== G3 FRAUD ANALYSIS ==========
Overall fraud rate: 0.0350
 network  cards  transactions  frauds  fraud_rate     lift
       1   5027        513596   19531    0.038028 1.086823
       2      7            30       0    0.000000 0.000000
       3      5            59       0    0.000000 0.000000
       4      5            59       1    0.016949 0.484400
       5      4           175       0    0.000000 0.000000
       6      4            39       2    0.051282 1.465620
       7      3            37       0    0.000000 0.000000
       8      3           189      55    0.291005 8.316811
       9      3            38       0    0.000000 0.000000
      10      3            41       0    0.000000 0.000000
      11      3            74       0    0.000000 0.000000
      12      3            25       0    0.000000 0.000000
      13      3             7       0    0.000000 0.000000
      14      3             9       0    0.000000 0.000000
      15      3            24       2    0.0833

In [21]:
# ============================================================
# STEP 7: STRONGEST CARD-CARD RELATIONSHIPS
# ============================================================

edges = []

for c1, c2, data in G.edges(data=True):

    shared = data["shared_count"]

    edges.append({
        "card1": c1,
        "card2": c2,
        "shared_count": shared,
        "shared_entities": data["shared_entities"]
    })

edge_df = pd.DataFrame(edges)

print("\n========== SHARED ENTITY STRENGTH ==========")

print(
    edge_df["shared_count"]
    .value_counts()
    .sort_index()
    .head(10)
)

print("\nTop 20 strongest relationships:")

print(
    edge_df
    .sort_values("shared_count", ascending=False)
    .head(20)
    .to_string(index=False)
)


========== SHARED ENTITY STRENGTH ==========
shared_count
2     420585
3     134646
4      62684
5      33306
6      19737
7      12538
8       8409
9       6056
10      4697
11      3577
Name: count, dtype: int64

Top 20 strongest relationships:
 card1  card2  shared_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [22]:
import pandas as pd
import networkx as nx
from collections import defaultdict
from itertools import combinations

# ============================================================
# CONFIG
# ============================================================

ENTITY_COLS = [
    "DeviceInfo",
    "card2",
    "id_20"
]

MAX_CARDS_PER_ENTITY = 1000
MIN_SHARED_TYPES = 2


# ============================================================
# STEP 1: ENTITY -> CARDS FOR EACH ENTITY TYPE
# ============================================================

entity_cards_by_type = {}

for entity_col in ENTITY_COLS:

    temp = (
        df[["card1", entity_col]]
        .dropna()
        .drop_duplicates()
    )

    # Number of unique cards using each entity value
    entity_card_counts = (
        temp.groupby(entity_col)["card1"]
        .nunique()
    )

    # Remove very generic entities
    valid_entities = set(
        entity_card_counts[
            entity_card_counts <= MAX_CARDS_PER_ENTITY
        ].index
    )

    temp = temp[
        temp[entity_col].isin(valid_entities)
    ]

    entity_cards = defaultdict(set)

    for entity, group in temp.groupby(entity_col):

        entity_cards[entity] = set(
            group["card1"]
        )

    entity_cards_by_type[entity_col] = entity_cards

    print(
        f"{entity_col}: "
        f"{len(entity_cards)} entities kept"
    )


# ============================================================
# STEP 2: FIND CARD PAIRS SHARING EACH ENTITY TYPE
# ============================================================

# pair -> set of entity TYPES shared
pair_types = defaultdict(set)

# pair -> evidence entities for each type
pair_evidence = defaultdict(lambda: defaultdict(set))


for entity_col in ENTITY_COLS:

    entity_cards = entity_cards_by_type[entity_col]

    for entity, cards in entity_cards.items():

        if len(cards) < 2:
            continue

        # Generate card pairs sharing THIS entity value
        for c1, c2 in combinations(sorted(cards), 2):

            pair = (c1, c2)

            # IMPORTANT:
            # We add the TYPE only once.
            pair_types[pair].add(entity_col)

            # Keep the actual value as evidence
            pair_evidence[pair][entity_col].add(entity)


# ============================================================
# STEP 3: KEEP ONLY MULTI-TYPE RELATIONSHIPS
# ============================================================

strong_pairs = {
    pair: types
    for pair, types in pair_types.items()
    if len(types) >= MIN_SHARED_TYPES
}


print("\n========== G3.1 RELATIONSHIPS ==========")

print("Total candidate card pairs:", len(pair_types))
print("Strong card pairs:", len(strong_pairs))


# ============================================================
# STEP 4: BUILD CORRECTED CARD GRAPH
# ============================================================

G = nx.Graph()

# Add all cards
cards = df["card1"].dropna().unique()

G.add_nodes_from(
    [(card, {"type": "Card"}) for card in cards]
)


for (c1, c2), types in strong_pairs.items():

    evidence = {}

    for entity_type in types:

        evidence[entity_type] = list(
            pair_evidence[(c1, c2)][entity_type]
        )

    G.add_edge(
        c1,
        c2,

        # Number of independent entity TYPES
        shared_types=len(types),

        # e.g. ["DeviceInfo", "card2"]
        shared_entity_types=list(types),

        # Actual values responsible for connection
        evidence=evidence
    )


# ============================================================
# STEP 5: GRAPH SUMMARY
# ============================================================

print("\n========== G3.1 GRAPH ==========")

print("Card nodes:", G.number_of_nodes())
print("Card-card edges:", G.number_of_edges())

components = list(nx.connected_components(G))

print("Connected components:", len(components))

largest = sorted(
    [len(c) for c in components],
    reverse=True
)[:20]

print("Largest components:", largest)


# ============================================================
# STEP 6: SHARED TYPE DISTRIBUTION
# ============================================================

shared_type_counts = defaultdict(int)

for _, _, data in G.edges(data=True):

    shared_type_counts[
        data["shared_types"]
    ] += 1


print("\nShared entity-type distribution:")

for k in sorted(shared_type_counts):

    print(
        f"{k} shared types: "
        f"{shared_type_counts[k]:,} edges"
    )

DeviceInfo: 1782 entities kept
card2: 498 entities kept
id_20: 389 entities kept

========== G3.1 RELATIONSHIPS ==========
Total candidate card pairs: 3721428
Strong card pairs: 235803

========== G3.1 GRAPH ==========
Card nodes: 13553
Card-card edges: 235803
Connected components: 9319
Largest components: [4105, 7, 7, 5, 5, 5, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]

Shared entity-type distribution:
2 shared types: 232,109 edges
3 shared types: 3,694 edges


In [27]:
# ============================================================
# STEP 7: FRAUD ANALYSIS
# ============================================================

card_stats = (
    df.groupby("card1")
      .agg(
          transactions=("TransactionID", "count"),
          frauds=("isFraud", "sum")
      )
)

card_stats["fraud_rate"] = (
    card_stats["frauds"] /
    card_stats["transactions"]
)

overall_fraud_rate = df["isFraud"].mean()

components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

results = []

for i, component in enumerate(components, 1):

    stats = card_stats.loc[
        card_stats.index.intersection(component)
    ]

    transactions = stats["transactions"].sum()
    frauds = stats["frauds"].sum()

    if transactions == 0:
        continue

    fraud_rate = frauds / transactions

    results.append({
        "network": i,
        "cards": len(component),
        "transactions": transactions,
        "frauds": frauds,
        "fraud_rate": fraud_rate,
        "lift": fraud_rate / overall_fraud_rate
    })


network_df = pd.DataFrame(results)


print("\n========== G3.1 FRAUD ANALYSIS ==========")

print(
    f"Overall fraud rate: "
    f"{overall_fraud_rate:.4f}"
)

print("\nTop 20 networks by size:")

print(
    network_df
    .sort_values("cards", ascending=False)
    .head(10)
    .to_string(index=False)
)


print("\nTop 20 networks by fraud rate "
      "(minimum 20 transactions):")

print(
    network_df[
        network_df["transactions"] >= 20
    ]
    .sort_values("fraud_rate", ascending=False)
    .head(10)
    .to_string(index=False)
)


========== G3.1 FRAUD ANALYSIS ==========
Overall fraud rate: 0.0350

Top 20 networks by size:
 network  cards  transactions  frauds  fraud_rate     lift
       1   4105        495708   19210    0.038753 1.107535
       2      7            30       0    0.000000 0.000000
       3      7           159       0    0.000000 0.000000
       4      5           146       0    0.000000 0.000000
       5      5            59       1    0.016949 0.484400
       6      5            59       0    0.000000 0.000000
       7      4           228       1    0.004386 0.125349
       8      4           175       0    0.000000 0.000000
       9      4            89       0    0.000000 0.000000
      10      4            39       2    0.051282 1.465620

Top 20 networks by fraud rate (minimum 20 transactions):
 network  cards  transactions  frauds  fraud_rate      lift
    4295      1            47      46    0.978723 27.971510
    1110      1            30      23    0.766667 21.911016
     253      1  

In [25]:
# ============================================================
# STEP 8: STRONGEST RELATIONSHIPS
# ============================================================

edge_rows = []

for c1, c2, data in G.edges(data=True):

    edge_rows.append({
        "card1": c1,
        "card2": c2,
        "shared_types": data["shared_types"],
        "shared_entity_types":
            data["shared_entity_types"],
        "evidence":
            data["evidence"]
    })

edge_df = pd.DataFrame(edge_rows)

print("\n========== STRONGEST RELATIONSHIPS ==========")

print(
    edge_df
    .sort_values(
        ["shared_types"],
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


========== STRONGEST RELATIONSHIPS ==========
 card1  card2  shared_types        shared_entity_types                                                                                                                                                                                                                                                                                                                                                                                                                                                            evidence
 16132   1257             3 [id_20, card2, DeviceInfo]                                                                                                                                                                                                                                                                                                                                                                                              {'id_20': 

In [29]:
import math
from collections import defaultdict

# ============================================================
# G3.2 — ENTITY RARITY
# ============================================================

ENTITY_COLS = [
    "DeviceInfo",
    "card2",
    "id_20"
]

MAX_CARDS_PER_ENTITY = 1000

# entity_card_count[(type, value)] = number of unique cards
entity_card_count = {}

for entity_col in ENTITY_COLS:

    temp = (
        df[["card1", entity_col]]
        .dropna()
        .drop_duplicates()
    )

    counts = (
        temp.groupby(entity_col)["card1"]
        .nunique()
    )

    for entity, count in counts.items():

        if count <= MAX_CARDS_PER_ENTITY:

            entity_card_count[
                (entity_col, entity)
            ] = count


print("Rarity dictionary size:",
      len(entity_card_count))

print("\nExample entity frequencies:")

for k, v in list(entity_card_count.items())[:15]:

    print(k, "->", v)

Rarity dictionary size: 2669

Example entity frequencies:
('DeviceInfo', '0PAJ5') -> 1
('DeviceInfo', '0PJA2') -> 1
('DeviceInfo', '0PM92') -> 4
('DeviceInfo', '1016S') -> 1
('DeviceInfo', '2PQ93') -> 1
('DeviceInfo', '2PS64 Build/NRD90M') -> 5
('DeviceInfo', '2PYB2') -> 4
('DeviceInfo', '2PZC5') -> 2
('DeviceInfo', '4003A') -> 1
('DeviceInfo', '4009F') -> 1
('DeviceInfo', '4013M Build/KOT49H') -> 3
('DeviceInfo', '4027A Build/KOT49H') -> 2
('DeviceInfo', '4034E') -> 1
('DeviceInfo', '4034G') -> 1
('DeviceInfo', '4047A Build/NRD90M') -> 4


In [30]:
# ============================================================
# G3.2 — RARITY-WEIGHTED EDGE SCORE
# ============================================================

N_CARDS = df["card1"].nunique()

for c1, c2, data in G.edges(data=True):

    total_rarity = 0.0
    evidence_details = []

    for entity_type in data["shared_entity_types"]:

        values = data["evidence"][entity_type]

        # Each type should normally have one or more
        # shared values.
        type_best_score = 0.0
        type_best_value = None
        type_best_count = None

        for value in values:

            count = entity_card_count.get(
                (entity_type, value)
            )

            if count is None:
                continue

            rarity = math.log(
                N_CARDS / count
            )

            if rarity > type_best_score:

                type_best_score = rarity
                type_best_value = value
                type_best_count = count

        total_rarity += type_best_score

        evidence_details.append({
            "type": entity_type,
            "value": type_best_value,
            "card_count": type_best_count,
            "rarity": type_best_score
        })

    data["rarity_score"] = total_rarity
    data["rarity_evidence"] = evidence_details

In [33]:
# ============================================================
# SHOW STRONGEST EDGES
# ============================================================

edge_rows = []

for c1, c2, data in G.edges(data=True):

    edge_rows.append({
        "card1": c1,
        "card2": c2,
        "shared_types": data["shared_types"],
        "rarity_score": data["rarity_score"],
        "evidence": data["rarity_evidence"]
    })

rarity_df = pd.DataFrame(edge_rows)

print("\n========== TOP RARITY-WEIGHTED RELATIONSHIPS ==========")

print(
    rarity_df
    .sort_values("rarity_score", ascending=False)
    .head(10)
    .to_string(index=False)
)


========== TOP RARITY-WEIGHTED RELATIONSHIPS ==========
 card1  card2  shared_types  rarity_score                                                                                                                                                                                                                                                                                 evidence
 10568  16136             3     23.285594                    [{'type': 'id_20', 'value': 384.0, 'card_count': 32, 'rarity': 6.048627301216289}, {'type': 'card2', 'value': 204.0, 'card_count': 3, 'rarity': 8.415750915347907}, {'type': 'DeviceInfo', 'value': 'A96 Build/LMY47I', 'card_count': 2, 'rarity': 8.82121602345607}]
 16655  14337             3     23.254823      [{'type': 'id_20', 'value': 275.0, 'card_count': 3, 'rarity': 8.415750915347907}, {'type': 'card2', 'value': 228.0, 'card_count': 11, 'rarity': 7.116467931217645}, {'type': 'DeviceInfo', 'value': 'SAMSUNG SM-A310F Build/NRD90M', 'card_count': 6, 'rar

In [34]:
print("\n========== SCORE DISTRIBUTION ==========")

print(
    rarity_df
    .groupby("shared_types")["rarity_score"]
    .describe()
)


========== SCORE DISTRIBUTION ==========
                 count       mean       std       min        25%        50%  \
shared_types                                                                  
2             232109.0   7.679031  1.644760  5.613082   6.465980   7.256873   
3               3694.0  12.970377  2.667923  9.303161  10.982119  12.313008   

                    75%        max  
shared_types                        
2              8.499087  17.642432  
3             14.236577  23.285594  


In [37]:
# ============================================================
# NETWORK RARITY + FRAUD
# ============================================================

components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

network_results = []

card_stats = (
    df.groupby("card1")
      .agg(
          transactions=("TransactionID", "count"),
          frauds=("isFraud", "sum")
      )
)

for network_id, component in enumerate(components, 1):

    cards = set(component)

    stats = card_stats.loc[
        card_stats.index.intersection(cards)
    ]

    transactions = stats["transactions"].sum()
    frauds = stats["frauds"].sum()

    if transactions == 0:
        continue

    fraud_rate = frauds / transactions

    # Edges inside this network
    subgraph = G.subgraph(cards)

    edge_scores = [
        data["rarity_score"]
        for _, _, data in subgraph.edges(data=True)
    ]

    if edge_scores:

        mean_rarity = sum(edge_scores) / len(edge_scores)
        max_rarity = max(edge_scores)

    else:

        mean_rarity = 0
        max_rarity = 0

    network_results.append({
        "network": network_id,
        "cards": len(cards),
        "transactions": transactions,
        "frauds": frauds,
        "fraud_rate": fraud_rate,
        "mean_edg_rar": mean_rarity,
        "max_edg_rar": max_rarity,
        "edges": subgraph.number_of_edges()
    })


network_rarity_df = pd.DataFrame(network_results)

print("\n========== NETWORK RARITY + FRAUD ==========")

print(
    network_rarity_df[
        (network_rarity_df["cards"] >= 2) &
        (network_rarity_df["transactions"] >= 20)
    ]
    .sort_values(
        ["fraud_rate", "mean_edg_rar"],
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


========== NETWORK RARITY + FRAUD ==========
 network  cards  transactions  frauds  fraud_rate  mean_edg_rar  max_edg_rar  edges
      15      3           189      55    0.291005      8.647763     8.647763      3
      52      2           109      20    0.183486     11.734349    11.734349      1
      28      3            24       2    0.083333      7.549182     7.549182      3
      44      2            31       2    0.064516     10.281692    10.281692      1
      10      4            39       2    0.051282      7.613193     7.613193      6
      11      3           255      13    0.050980     11.727579    11.727579      3
       1   4105        495708   19210    0.038753      7.760619    23.285594 235608
      30      2            91       3    0.032967      7.099990     7.099990      1
      36      2           137       4    0.029197     11.503086    11.503086      1
      19      3           183       5    0.027322     11.825321    11.825321      3
      62      2           144 

using community detection on our updated G3 to find the suspicious networks. Finding these networks solves the problem

In [38]:
# ============================================================
# G3.3 — WEIGHTED COMMUNITY DETECTION
# ============================================================

import networkx as nx
import pandas as pd

print("Running Louvain community detection...")

communities = nx.community.louvain_communities(
    G,
    weight="rarity_score",
    resolution=1.0,
    seed=42
)

print("\n========== COMMUNITY DETECTION ==========")

print("Total communities:", len(communities))

community_sizes = sorted(
    [len(c) for c in communities],
    reverse=True
)

print("Largest communities:")
print(community_sizes[:15])

Running Louvain community detection...

========== COMMUNITY DETECTION ==========
Total communities: 9367
Largest communities:
[1554, 1058, 607, 212, 182, 70, 69, 66, 23, 22, 20, 18, 17, 17, 14]


In [40]:
# ============================================================
# COMMUNITY FRAUD ANALYSIS
# ============================================================

card_stats = (
    df.groupby("card1")
      .agg(
          transactions=("TransactionID", "count"),
          frauds=("isFraud", "sum")
      )
)

card_stats["fraud_rate"] = (
    card_stats["frauds"] /
    card_stats["transactions"]
)

overall_fraud_rate = df["isFraud"].mean()

community_results = []

for community_id, community in enumerate(
    communities, 1
):

    stats = card_stats.loc[
        card_stats.index.intersection(community)
    ]

    transactions = stats["transactions"].sum()
    frauds = stats["frauds"].sum()

    if transactions == 0:
        continue

    fraud_rate = frauds / transactions

    subgraph = G.subgraph(community)

    # Internal edge strength
    weights = [
        data["rarity_score"]
        for _, _, data in subgraph.edges(data=True)
    ]

    mean_edge_weight = (
        sum(weights) / len(weights)
        if weights else 0
    )

    community_results.append({
        "community": community_id,
        "cards": len(community),
        "trans": transactions,
        "frauds": frauds,
        "fraud_rate": fraud_rate,
        "lift": fraud_rate / overall_fraud_rate,
        "int_edges": subgraph.number_of_edges(),
        "mean_edg_wt": mean_edge_weight
    })


community_df = pd.DataFrame(
    community_results
)

print("\n========== TOP COMMUNITIES BY FRAUD RATE ==========")

print(
    community_df[
        (community_df["cards"] >= 2) &
        (community_df["trans"] >= 20)
    ]
    .sort_values(
        "fraud_rate",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)


========== TOP COMMUNITIES BY FRAUD RATE ==========
 community  cards  trans  frauds  fraud_rate     lift  int_edges  mean_edg_wt
       123      3    189      55    0.291005 8.316811          3     8.647763
      1027      2    109      20    0.183486 5.243961          1    11.734349
       378   1058  66678    7697    0.115435 3.299095      28300     9.979607
      2306      3     24       2    0.083333 2.381632          3     7.549182
      5153      3     25       2    0.080000 2.286367          3    10.238001
       308      3     71       5    0.070423 2.012647          3    10.129132
       454      2     31       2    0.064516 1.843844          1    10.281692
      1002      4     47       3    0.063830 1.824229          6     7.296762
      4817      4     39       2    0.051282 1.465620          6     7.613193
      4543      3    255      13    0.050980 1.456999          3    11.727579
      8940     23    397      17    0.042821 1.223811         63     8.631122
      1233 

In [43]:
# ============================================================
# RING CANDIDATE SCORE
# ============================================================

community_df["density"] = (
    2 * community_df["int_edges"]
    /
    (
        community_df["cards"]
        *
        (community_df["cards"] - 1)
    )
).fillna(0)


community_df["ring_score"] = (
    community_df["fraud_rate"]
    *
    community_df["mean_edg_wt"]
    *
    (1 + community_df["density"])
    *
    __import__("numpy").log1p(
        community_df["trans"]
    )
)


print("\n========== TOP RING CANDIDATES ==========")

print(
    community_df[
        (community_df["cards"] >= 2) &
        (community_df["frauds"] >= 2)
    ]
    .sort_values(
        "ring_score",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)


========== TOP RING CANDIDATES ==========
 community  cards  trans  frauds  fraud_rate     lift  int_edges  mean_edg_wt  density  ring_score
       123      3    189      55    0.291005 8.316811          3     8.647763 1.000000   26.408743
      1027      2    109      20    0.183486 5.243961          1    11.734349 1.000000   20.241129
       378   1058  66678    7697    0.115435 3.299095      28300     9.979607 0.050612   13.443638
      4318      2      9       2    0.222222 6.351019          1     8.203483 1.000000    8.395208
      4655      2     13       2    0.153846 4.396859          1     8.362099 1.000000    6.790172
      4543      3    255      13    0.050980 1.456999          3    11.727579 1.000000    6.630663
       308      3     71       5    0.070423 2.012647          3    10.129132 1.000000    6.101256
      5153      3     25       2    0.080000 2.286367          3    10.238001 1.000000    5.337023
       454      2     31       2    0.064516 1.843844          1  

In [46]:
# ============================================================
# INSPECT TOP COMMUNITY
# ============================================================

top_community_id = (
    community_df[
        (community_df["cards"] >= 2) &
        (community_df["frauds"] >= 2)
    ]
    .sort_values(
        "ring_score",
        ascending=False
    )
    .iloc[0]["community"]
)

top_community = communities[
    int(top_community_id) - 1
]

print("\n========== TOP RING ==========")

print("Community:", top_community_id)
print("Cards:", len(top_community))

print("\nCards:")

for card in sorted(top_community):
    print(card)


print("\n========== RELATIONSHIPS ==========")

subgraph = G.subgraph(top_community)

for c1, c2, data in sorted(
    subgraph.edges(data=True),
    key=lambda x: x[2]["rarity_score"],
    reverse=True
)[:20]:

    print(
        f"\n{c1} <--> {c2}"
    )

    print(
        "Shared types:",
        data["shared_entity_types"]
    )

    print(
        "Rarity score:",
        round(data["rarity_score"], 3)
    )

    print(
        "Evidence:",
        data["evidence"]
    )


========== TOP RING ==========
Community: 123.0
Cards: 3

Cards:
2508
3331
16927

========== RELATIONSHIPS ==========

3331 <--> 2508
Shared types: ['id_20', 'card2']
Rarity score: 8.648
Evidence: {'id_20': [181.0], 'card2': [310.0]}

3331 <--> 16927
Shared types: ['id_20', 'card2']
Rarity score: 8.648
Evidence: {'id_20': [181.0], 'card2': [310.0]}

2508 <--> 16927
Shared types: ['id_20', 'card2']
Rarity score: 8.648
Evidence: {'id_20': [181.0], 'card2': [310.0]}


In [49]:
# ============================================================
# G3.4 — TEMPORAL COORDINATION
# ============================================================

import numpy as np
import pandas as pd

TIME_WINDOW = 24 * 60 * 60   # 24 hours in TransactionDT units


# ============================================================
# STEP 1: PREPARE TRANSACTION TIMES PER CARD
# ============================================================

card_times = {}

for card, group in df.groupby("card1"):

    card_times[card] = np.sort(
        group["TransactionDT"]
        .dropna()
        .values
    )


# ============================================================
# STEP 2: FUNCTION — TEMPORAL PROXIMITY BETWEEN TWO CARDS
# ============================================================

def temporal_proximity(times_a, times_b, window=TIME_WINDOW):

    if len(times_a) == 0 or len(times_b) == 0:
        return 0.0

    # For every transaction of A, find nearest transaction of B
    positions = np.searchsorted(
        times_b,
        times_a
    )

    nearest = np.full(
        len(times_a),
        np.inf
    )

    # Candidate on right
    valid_right = positions < len(times_b)

    nearest[valid_right] = np.minimum(
        nearest[valid_right],
        np.abs(
            times_a[valid_right]
            -
            times_b[positions[valid_right]]
        )
    )

    # Candidate on left
    valid_left = positions > 0

    nearest[valid_left] = np.minimum(
        nearest[valid_left],
        np.abs(
            times_a[valid_left]
            -
            times_b[positions[valid_left] - 1]
        )
    )

    # Fraction of A transactions that have
    # another card's transaction within the window
    return np.mean(nearest <= window)


# ============================================================
# STEP 3: CALCULATE TEMPORAL COORDINATION FOR COMMUNITIES
# ============================================================

community_temporal = []

for community_id, community in enumerate(
    communities,
    1
):

    cards = list(community)

    # Ignore isolated/single-card communities
    if len(cards) < 2:
        continue

    # --------------------------------------------------------
    # Pairwise temporal proximity
    # --------------------------------------------------------

    pair_scores = []

    for i in range(len(cards)):

        for j in range(i + 1, len(cards)):

            c1 = cards[i]
            c2 = cards[j]

            score_1 = temporal_proximity(
                card_times.get(c1, []),
                card_times.get(c2, [])
            )

            score_2 = temporal_proximity(
                card_times.get(c2, []),
                card_times.get(c1, [])
            )

            # Symmetric score
            pair_score = (
                score_1 + score_2
            ) / 2

            pair_scores.append(pair_score)

    if pair_scores:

        temporal_score = np.mean(
            pair_scores
        )

        max_pair_temporal = np.max(
            pair_scores
        )

    else:

        temporal_score = 0
        max_pair_temporal = 0


    # --------------------------------------------------------
    # Fraud participation
    # --------------------------------------------------------

    stats = card_stats.loc[
        card_stats.index.intersection(cards)
    ]

    fraud_cards = (
        stats["frauds"] > 0
    ).sum()

    fraud_participation = (
        fraud_cards / len(cards)
    )

    total_frauds = stats["frauds"].sum()
    total_transactions = stats["transactions"].sum()

    fraud_rate = (
        total_frauds / total_transactions
        if total_transactions > 0
        else 0
    )


    community_temporal.append({

        "community": community_id,

        "cards": len(cards),

        "transactions":
            total_transactions,

        "frauds":
            total_frauds,

        "fraud_rate":
            fraud_rate,

        "fraud_cards":
            fraud_cards,

        "fraud_participation":
            fraud_participation,

        "temporal_coordination":
            temporal_score,

        "max_pair_temporal":
            max_pair_temporal
    })


temporal_df = pd.DataFrame(
    community_temporal
)


print("\n========== TEMPORAL COORDINATION ==========")

print(
    temporal_df[
        (temporal_df["cards"] >= 2) &
        (temporal_df["transactions"] >= 20)
    ]
    .sort_values(
        "temporal_coordination",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)


========== TEMPORAL COORDINATION ==========
 community  cards  transactions  frauds  fraud_rate  fraud_cards  fraud_participation  temporal_coordination  max_pair_temporal
       546      2           245       0    0.000000            0             0.000000               0.720443           0.720443
       372      2           121       0    0.000000            0             0.000000               0.582609           0.582609
      4226      3           438       2    0.004566            1             0.333333               0.580415           0.732432
       311      2           213       0    0.000000            0             0.000000               0.548876           0.548876
        77      3           160       3    0.018750            1             0.333333               0.525381           0.550000
      6381      2            86       0    0.000000            0             0.000000               0.523810           0.523810
       279      2           122       0    0.000000        

In [50]:
# ============================================================
# STEP 4: INSPECT COMMUNITY 123
# ============================================================

TARGET_COMMUNITY = 123

row = temporal_df[
    temporal_df["community"] == TARGET_COMMUNITY
]

print("\n========== COMMUNITY 123 ==========")

print(
    row.to_string(index=False)
)


cards = list(
    communities[TARGET_COMMUNITY - 1]
)

print("\nPairwise temporal proximity:")

for i in range(len(cards)):

    for j in range(i + 1, len(cards)):

        c1 = cards[i]
        c2 = cards[j]

        score_1 = temporal_proximity(
            card_times[c1],
            card_times[c2]
        )

        score_2 = temporal_proximity(
            card_times[c2],
            card_times[c1]
        )

        print(
            f"{c1} <--> {c2}: "
            f"{(score_1 + score_2) / 2:.4f}"
        )


========== COMMUNITY 123 ==========
 community  cards  transactions  frauds  fraud_rate  fraud_cards  fraud_participation  temporal_coordination  max_pair_temporal
       123      3           189      55    0.291005            1             0.333333                0.50155           0.511111

Pairwise temporal proximity:
3331 <--> 2508: 0.5111
3331 <--> 16927: 0.4884
2508 <--> 16927: 0.5051


In [52]:
# ============================================================
# G4.1 — DEVICEINFO + CARD2 + ID_20 + ID_19
# ============================================================

import pandas as pd
import networkx as nx
from collections import defaultdict
from itertools import combinations

ENTITY_COLS = [
    "DeviceInfo",
    "card2",
    "id_20",
    "id_19"
]

MAX_CARDS_PER_ENTITY = 1000
MIN_SHARED_TYPES = 2


# ============================================================
# 1. ENTITY -> CARDS
# ============================================================

entity_cards_by_type = {}

for entity_col in ENTITY_COLS:

    temp = (
        df[["card1", entity_col]]
        .dropna()
        .drop_duplicates()
    )

    entity_card_counts = (
        temp.groupby(entity_col)["card1"]
        .nunique()
    )

    valid_entities = set(
        entity_card_counts[
            entity_card_counts <= MAX_CARDS_PER_ENTITY
        ].index
    )

    temp = temp[
        temp[entity_col].isin(valid_entities)
    ]

    entity_cards = defaultdict(set)

    for entity, group in temp.groupby(entity_col):

        entity_cards[entity] = set(
            group["card1"]
        )

    entity_cards_by_type[entity_col] = entity_cards

    print(
        f"{entity_col}: "
        f"{len(entity_cards)} entities kept"
    )


# ============================================================
# 2. CARD PAIRS + SHARED ENTITY TYPES
# ============================================================

pair_types = defaultdict(set)

pair_evidence = defaultdict(
    lambda: defaultdict(set)
)


for entity_col in ENTITY_COLS:

    entity_cards = entity_cards_by_type[entity_col]

    for entity, cards in entity_cards.items():

        if len(cards) < 2:
            continue

        for c1, c2 in combinations(
            sorted(cards), 2
        ):

            pair = (c1, c2)

            # IMPORTANT:
            # entity TYPE is counted only once
            pair_types[pair].add(
                entity_col
            )

            pair_evidence[
                pair
            ][entity_col].add(entity)


# ============================================================
# 3. KEEP MULTI-TYPE RELATIONSHIPS
# ============================================================

strong_pairs = {
    pair: types
    for pair, types in pair_types.items()
    if len(types) >= MIN_SHARED_TYPES
}


print("\n========== G4 RELATIONSHIPS ==========")

print(
    "Total candidate card pairs:",
    len(pair_types)
)

print(
    "Strong card pairs:",
    len(strong_pairs)
)


# ============================================================
# 4. BUILD GRAPH
# ============================================================

G4 = nx.Graph()

cards = df["card1"].dropna().unique()

G4.add_nodes_from(
    [
        (card, {"type": "Card"})
        for card in cards
    ]
)


for (c1, c2), types in strong_pairs.items():

    evidence = {}

    for entity_type in types:

        evidence[entity_type] = list(
            pair_evidence[
                (c1, c2)
            ][entity_type]
        )

    G4.add_edge(
        c1,
        c2,

        shared_types=len(types),

        shared_entity_types=list(types),

        evidence=evidence
    )


# ============================================================
# 5. GRAPH SUMMARY
# ============================================================

print("\n========== G4 GRAPH ==========")

print(
    "Card nodes:",
    G4.number_of_nodes()
)

print(
    "Card-card edges:",
    G4.number_of_edges()
)

components = list(
    nx.connected_components(G4)
)

print(
    "Connected components:",
    len(components)
)

largest = sorted(
    [len(c) for c in components],
    reverse=True
)[:20]

print(
    "Largest components:",
    largest
)


# ============================================================
# 6. SHARED TYPE DISTRIBUTION
# ============================================================

shared_type_counts = defaultdict(int)

for _, _, data in G4.edges(data=True):

    shared_type_counts[
        data["shared_types"]
    ] += 1


print(
    "\nShared entity-type distribution:"
)

for k in sorted(shared_type_counts):

    print(
        f"{k} shared types: "
        f"{shared_type_counts[k]:,} edges"
    )

DeviceInfo: 1782 entities kept
card2: 498 entities kept
id_20: 389 entities kept
id_19: 514 entities kept

========== G4 RELATIONSHIPS ==========
Total candidate card pairs: 4883369
Strong card pairs: 1187069

========== G4 GRAPH ==========
Card nodes: 13553
Card-card edges: 1187069
Connected components: 7309
Largest components: [6237, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Shared entity-type distribution:
2 shared types: 1,012,825 edges
3 shared types: 170,921 edges
4 shared types: 3,323 edges


In [53]:
# ============================================================
# G4.2 — RARITY WEIGHTING
# ============================================================

import math

N_CARDS = df["card1"].nunique()

entity_card_count_g4 = {}

for entity_col in ENTITY_COLS:

    temp = (
        df[["card1", entity_col]]
        .dropna()
        .drop_duplicates()
    )

    counts = (
        temp.groupby(entity_col)["card1"]
        .nunique()
    )

    for entity, count in counts.items():

        if count <= MAX_CARDS_PER_ENTITY:

            entity_card_count_g4[
                (entity_col, entity)
            ] = count


# ============================================================
# APPLY RARITY TO G4 EDGES
# ============================================================

for c1, c2, data in G4.edges(data=True):

    total_rarity = 0.0
    evidence_details = []

    for entity_type in data["shared_entity_types"]:

        values = data["evidence"][entity_type]

        best_score = 0.0
        best_value = None
        best_count = None

        for value in values:

            count = entity_card_count_g4.get(
                (entity_type, value)
            )

            if count is None:
                continue

            rarity = math.log(
                N_CARDS / count
            )

            if rarity > best_score:

                best_score = rarity
                best_value = value
                best_count = count

        total_rarity += best_score

        evidence_details.append({
            "type": entity_type,
            "value": best_value,
            "card_count": best_count,
            "rarity": best_score
        })

    data["rarity_score"] = total_rarity
    data["rarity_evidence"] = evidence_details


print("G4 rarity scoring complete.")

G4 rarity scoring complete.


In [54]:
# ============================================================
# G4.2 — RARITY DISTRIBUTION
# ============================================================

edge_rows = []

for c1, c2, data in G4.edges(data=True):

    edge_rows.append({
        "card1": c1,
        "card2": c2,
        "shared_types": data["shared_types"],
        "rarity_score": data["rarity_score"]
    })

g4_edge_df = pd.DataFrame(edge_rows)

print("\n========== G4 RARITY DISTRIBUTION ==========")

print(
    g4_edge_df
    .groupby("shared_types")["rarity_score"]
    .describe()
)


========== G4 RARITY DISTRIBUTION ==========
                  count       mean       std        min        25%        50%  \
shared_types                                                                    
2             1012825.0   6.846966  1.080460   5.311606   6.107618   6.620236   
3              170921.0  11.740624  2.347886   8.272036   9.972593  11.151970   
4                3323.0  17.428031  3.526307  12.137425  14.753297  16.673479   

                    75%        max  
shared_types                        
2              7.306935  17.236967  
3             13.010083  25.770501  
4             19.389567  31.031915  


In [55]:
print("\n========== TOP G4 RELATIONSHIPS ==========")

print(
    g4_edge_df
    .sort_values(
        "rarity_score",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)


========== TOP G4 RELATIONSHIPS ==========
 card1  card2  shared_types  rarity_score
  4461  13832             4     31.031915
 16655  14337             4     30.977426
 13832  15257             4     30.619069
  4461  15257             4     30.213604
  3154   5812             4     30.043117
  9633   2801             4     29.568659
 15885   2256             4     29.532291
 16489   3683             4     29.532291
  4504  16062             4     29.467753
  2803  11106             4     29.313602
 15885   1976             4     29.309148
 15885   9026             4     29.309148
 10568  16136             4     29.303450
  9633   5009             4     28.433679
 16659   8695             4     28.311789


In [ ]:
# ============================================================
# G4.3 — WEIGHTED COMMUNITY DETECTION
# ============================================================

print("Running G4 Louvain...")

communities_g4 = nx.community.louvain_communities(
    G4,
    weight="rarity_score",
    resolution=1.0,
    seed=42
)

print("\n========== G4 COMMUNITIES ==========")

print(
    "Total communities:",
    len(communities_g4)
)

community_sizes_g4 = sorted(
    [len(c) for c in communities_g4],
    reverse=True
)

print(
    "Largest communities:"
)

print(
    community_sizes_g4[:30]
)

Running G4 Louvain...

========== G4 COMMUNITIES ==========
Total communities: 7346
Largest communities:
[3966, 1570, 562, 23, 8, 8, 7, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2]


In [57]:
# ============================================================
# G4.3 — COMMUNITY FRAUD ANALYSIS
# ============================================================

card_stats = (
    df.groupby("card1")
      .agg(
          transactions=("TransactionID", "count"),
          frauds=("isFraud", "sum")
      )
)

card_stats["fraud_rate"] = (
    card_stats["frauds"] /
    card_stats["transactions"]
)

overall_fraud_rate = df["isFraud"].mean()

g4_results = []

for community_id, community in enumerate(
    communities_g4,
    1
):

    cards = list(community)

    if len(cards) < 2:
        continue

    stats = card_stats.loc[
        card_stats.index.intersection(cards)
    ]

    transactions = stats["transactions"].sum()
    frauds = stats["frauds"].sum()

    if transactions == 0:
        continue

    fraud_rate = frauds / transactions

    fraud_cards = (
        stats["frauds"] > 0
    ).sum()

    fraud_participation = (
        fraud_cards / len(cards)
    )

    subgraph = G4.subgraph(cards)

    weights = [
        data["rarity_score"]
        for _, _, data
        in subgraph.edges(data=True)
    ]

    mean_rarity = (
        np.mean(weights)
        if weights else 0
    )

    max_rarity = (
        np.max(weights)
        if weights else 0
    )

    g4_results.append({

        "community": community_id,

        "cards": len(cards),

        "transactions": transactions,

        "frauds": frauds,

        "fraud_rate": fraud_rate,

        "lift":
            fraud_rate / overall_fraud_rate,

        "fraud_cards":
            fraud_cards,

        "fraud_participation":
            fraud_participation,

        "internal_edges":
            subgraph.number_of_edges(),

        "mean_rarity":
            mean_rarity,

        "max_rarity":
            max_rarity
    })


g4_df = pd.DataFrame(g4_results)


print(
    "\n========== TOP G4 COMMUNITIES =========="
)

print(
    g4_df[
        (g4_df["cards"] >= 2) &
        (g4_df["transactions"] >= 20)
    ]
    .sort_values(
        ["fraud_rate", "fraud_participation"],
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)


========== TOP G4 COMMUNITIES ==========
 community  cards  transactions  frauds  fraud_rate     lift  fraud_cards  fraud_participation  internal_edges  mean_rarity  max_rarity
       271   1570         70292    7833    0.111435 3.184771          364             0.231847          231962     8.033117   31.031915
       242      8           218      19    0.087156 2.490881            1             0.125000              17     7.075612    7.578613
      3030      3            23       1    0.043478 1.242591            1             0.333333               3     8.702425    8.702425
      1480      5           166       5    0.030120 0.860831            1             0.200000               8    11.519707   15.813236
      1205   3966        399940   10490    0.026229 0.749612          869             0.219112          577589     7.301942   29.313602
      1099    562         56554    1473    0.026046 0.744381          188             0.334520           76371     7.401520   20.155773
      

In [58]:
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict


# ============================================================
# 1. TRANSACTION TIME INDEX
# ============================================================

fraud_tx = df[df["isFraud"] == 1].copy()

fraud_times = (
    fraud_tx
    .groupby("card1")["TransactionDT"]
    .apply(lambda x: np.sort(x.values))
    .to_dict()
)

fraud_card_set = set(fraud_times.keys())

print("Fraudulent cards:", len(fraud_card_set))


# ============================================================
# 2. TEMPORAL COORDINATION
# ============================================================

def temporal_proximity(times_a, times_b, window=86400):
    """
    Fraction of transactions from A that occur within
    `window` seconds of at least one transaction from B.
    """

    if len(times_a) == 0 or len(times_b) == 0:
        return 0.0

    idx = np.searchsorted(times_b, times_a)

    nearest = np.full(len(times_a), np.inf)

    # Right neighbour
    valid = idx < len(times_b)
    nearest[valid] = np.abs(
        times_b[idx[valid]] - times_a[valid]
    )

    # Left neighbour
    valid_left = idx > 0
    nearest[valid_left] = np.minimum(
        nearest[valid_left],
        np.abs(
            times_b[idx[valid_left] - 1] -
            times_a[valid_left]
        )
    )

    return np.mean(nearest <= window)


def symmetric_temporal_proximity(times_a, times_b, window=86400):

    ab = temporal_proximity(times_a, times_b, window)
    ba = temporal_proximity(times_b, times_a, window)

    return (ab + ba) / 2

Fraudulent cards: 1740


In [59]:
# ============================================================
# 3. COMMUNITY → CARDS
# ============================================================

community_cards = {
    i: list(comm)
    for i, comm in enumerate(communities_g4)
}


# ============================================================
# 4. FRAUD-TO-FRAUD TEMPORAL COORDINATION
# ============================================================

temporal_results = []

for community_id, cards in community_cards.items():

    # Only fraudulent cards
    fraud_cards = [
        c for c in cards
        if c in fraud_card_set
    ]

    # Need at least two fraudulent members
    if len(fraud_cards) < 2:
        temporal_results.append({
            "community": community_id,
            "fraud_cards": len(fraud_cards),
            "fraud_temporal_edges": 0,
            "mean_fraud_temporal": 0.0,
            "max_fraud_temporal": 0.0
        })
        continue

    scores = []

    for i in range(len(fraud_cards)):
        for j in range(i + 1, len(fraud_cards)):

            a = fraud_cards[i]
            b = fraud_cards[j]

            score = symmetric_temporal_proximity(
                fraud_times[a],
                fraud_times[b],
                window=86400
            )

            scores.append(score)

    if scores:
        mean_temporal = np.mean(scores)
        max_temporal = np.max(scores)
        strong_temporal_edges = sum(
            s >= 0.5 for s in scores
        )
    else:
        mean_temporal = 0.0
        max_temporal = 0.0
        strong_temporal_edges = 0

    temporal_results.append({
        "community": community_id,
        "fraud_cards": len(fraud_cards),
        "fraud_temporal_edges": strong_temporal_edges,
        "mean_fraud_temporal": mean_temporal,
        "max_fraud_temporal": max_temporal
    })


temporal_df = pd.DataFrame(temporal_results)

print(
    temporal_df
    .sort_values(
        ["mean_fraud_temporal", "fraud_cards"],
        ascending=False
    )
    .head(15)
)

      community  fraud_cards  fraud_temporal_edges  mean_fraud_temporal  \
270         270          364                  8169             0.098189   
1204       1204          869                 26990             0.063470   
1098       1098          188                   831             0.048216   
1932       1932            2                     0             0.000000   
12           12            1                     0             0.000000   
13           13            1                     0             0.000000   
14           14            1                     0             0.000000   
38           38            1                     0             0.000000   
41           41            1                     0             0.000000   
46           46            1                     0             0.000000   
47           47            1                     0             0.000000   
71           71            1                     0             0.000000   
80           80          

In [63]:
# ============================================================
# 5. MERGE TEMPORAL SIGNAL
# ============================================================

ring_df = g4_df.merge(
    temporal_df[
        [
            "community",
            "fraud_temporal_edges",
            "mean_fraud_temporal",
            "max_fraud_temporal"
        ]
    ],
    on="community",
    how="left"
)

ring_df[
    [
        "community",
        "cards",
        "transactions",
        "frauds",
        "fraud_rate",
        "lift",
        "fraud_cards",
        "fraud_participation",
        "mean_rarity",
        "max_rarity",
        "mean_fraud_temporal",
        "max_fraud_temporal",
        "fraud_temporal_edges"
    ]
].sort_values(
    "mean_fraud_temporal",
    ascending=False
).head()

,community,cards,transactions,frauds,fraud_rate,lift,fraud_cards,fraud_participation,mean_rarity,max_rarity,mean_fraud_temporal,max_fraud_temporal,fraud_temporal_edges
0,13,2,13,2,0.153846,4.396859,1,0.500000,8.362099,8.362099,0.0,0.0,0
1,154,4,122,0,0.000000,0.000000,0,0.000000,6.783558,6.783558,0.0,0.0,0
2,242,8,218,19,0.087156,2.490881,1,0.125000,7.075612,7.578613,0.0,0.0,0
3,271,1570,70292,7833,0.111435,3.184771,364,0.231847,8.033117,31.031915,0.0,0.0,0
4,402,4,22,0,0.000000,0.000000,0,0.000000,7.898527,7.898527,0.0,0.0,0


In [64]:
# ============================================================
# 6. NORMALIZED COMPONENTS
# ============================================================

ring_df["rarity_component"] = np.log1p(
    ring_df["mean_rarity"]
)

ring_df["fraud_component"] = (
    ring_df["fraud_participation"]
)

ring_df["temporal_component"] = (
    ring_df["mean_fraud_temporal"]
)

ring_df["density"] = (
    2 * ring_df["internal_edges"]
    / (
        ring_df["cards"] *
        (ring_df["cards"] - 1)
    ).replace(0, np.nan)
)

ring_df["density"] = ring_df["density"].fillna(0)


# ============================================================
# 7. RING SCORE
# ============================================================

ring_df["ring_score"] = (
    ring_df["rarity_component"]
    * ring_df["fraud_component"]
    * ring_df["temporal_component"]
    * (1 + ring_df["density"])
    * np.log1p(ring_df["frauds"])
)

In [65]:
top_rings = (
    ring_df[
        (ring_df["fraud_cards"] >= 2) &
        (ring_df["fraud_temporal_edges"] >= 1)
    ]
    .sort_values(
        "ring_score",
        ascending=False
    )
)

print("\n========== TOP CANDIDATE ABUSE RINGS ==========\n")

print(
    top_rings[
        [
            "community",
            "cards",
            "transactions",
            "frauds",
            "fraud_rate",
            "lift",
            "fraud_cards",
            "fraud_participation",
            "mean_rarity",
            "max_rarity",
            "mean_fraud_temporal",
            "max_fraud_temporal",
            "fraud_temporal_edges",
            "ring_score"
        ]
    ].head(10).to_string(index=False)
)


========== TOP CANDIDATE ABUSE RINGS ==========

Empty DataFrame
Columns: [community, cards, transactions, frauds, fraud_rate, lift, fraud_cards, fraud_participation, mean_rarity, max_rarity, mean_fraud_temporal, max_fraud_temporal, fraud_temporal_edges, ring_score]
Index: []


In [66]:
# ============================================================
# 8. BUILD FRAUD-CORE GRAPH
# ============================================================

def build_fraud_core(G, fraud_times, rarity_threshold=8.0,
                     temporal_threshold=0.30):

    fraud_core = nx.Graph()

    # Only fraudulent cards that actually exist in G
    fraud_nodes = [
        c for c in G.nodes
        if c in fraud_times
    ]

    fraud_core.add_nodes_from(fraud_nodes)

    fraud_set = set(fraud_nodes)

    for u, v, data in G.edges(data=True):

        # Both endpoints must be fraudulent
        if u not in fraud_set or v not in fraud_set:
            continue

        rarity = data.get("rarity_score", 0)

        if rarity < rarity_threshold:
            continue

        temporal = symmetric_temporal_proximity(
            fraud_times[u],
            fraud_times[v],
            window=86400
        )

        if temporal < temporal_threshold:
            continue

        fraud_core.add_edge(
            u,
            v,
            rarity_score=rarity,
            temporal_score=temporal,
            shared_types=data.get(
                "shared_types", 0
            ),
            shared_entity_types=data.get(
                "shared_entity_types", []
            ),
            evidence=data.get(
                "evidence", {}
            )
        )

    return fraud_core


fraud_core = build_fraud_core(
    G4,
    fraud_times,
    rarity_threshold=8.0,
    temporal_threshold=0.30
)

print("Fraud-core nodes:", fraud_core.number_of_nodes())
print("Fraud-core edges:", fraud_core.number_of_edges())

Fraud-core nodes: 1740
Fraud-core edges: 33512


In [70]:
# ============================================================
# 9. CANDIDATE FRAUD RINGS
# ============================================================

candidate_rings = []

for ring_id, component in enumerate(
    nx.connected_components(fraud_core)
):

    if len(component) < 2:
        continue

    subgraph = fraud_core.subgraph(component)

    candidate_rings.append({
        "ring_id": ring_id,
        "cards": len(component),
        "edges": subgraph.number_of_edges(),
        "density": nx.density(subgraph),
        "mean_rarity": np.mean([
            d["rarity_score"]
            for _, _, d in subgraph.edges(data=True)
        ]),
        "mean_temporal": np.mean([
            d["temporal_score"]
            for _, _, d in subgraph.edges(data=True)
        ])
    })


candidate_rings_df = pd.DataFrame(candidate_rings)

candidate_rings_df = candidate_rings_df.sort_values(
    ["cards", "mean_temporal", "mean_rarity"],
    ascending=False
)

print("\n========== FRAUD CORE RINGS ==========\n")

print(
    candidate_rings_df
    .head(10)
    .to_string(index=False)
)


========== FRAUD CORE RINGS ==========

 ring_id  cards  edges  density  mean_rarity  mean_temporal
       0   1228  33511 0.044481    12.753010        0.48219
     102      2      1 1.000000     9.348382        1.00000


In [71]:
def explain_ring(G, fraud_core, ring_cards):

    explanations = []

    ring_set = set(ring_cards)

    for u, v, data in fraud_core.edges():

        if u not in ring_set or v not in ring_set:
            continue

        explanations.append({
            "card_a": u,
            "card_b": v,
            "shared_types": data["shared_types"],
            "rarity": data["rarity_score"],
            "temporal_coordination": data["temporal_score"],
            "evidence": data["evidence"]
        })

    return pd.DataFrame(explanations)


# Example:
# ring_cards = list(nx.connected_components(fraud_core))[0]
# explain_ring(G4, fraud_core, ring_cards)

In [72]:
# ============================================================
# INSPECT A FRAUD CORE RING
# ============================================================

ring_id = 102

components = list(nx.connected_components(fraud_core))

ring_cards = list(components[ring_id])

print("Cards:", ring_cards)

ring_subgraph = fraud_core.subgraph(ring_cards)

for u, v, data in ring_subgraph.edges(data=True):

    print("\n--------------------------------")
    print("Card A:", u)
    print("Card B:", v)
    print("Rarity:", data["rarity_score"])
    print("Temporal:", data["temporal_score"])
    print("Shared types:", data["shared_entity_types"])
    print("Evidence:", data["evidence"])

Cards: [8936, np.int64(11698)]

--------------------------------
Card A: 8936
Card B: 11698
Rarity: 9.348382406810114
Temporal: 1.0
Shared types: ['id_20', 'card2']
Evidence: {'id_20': [277.0], 'card2': [383.0]}


In [77]:
g4_df.head()

,community,cards,transactions,frauds,fraud_rate,lift,fraud_cards,fraud_participation,internal_edges,mean_rarity,max_rarity
0,13,2,13,2,0.153846,4.396859,1,0.500000,1,8.362099,8.362099
1,154,4,122,0,0.000000,0.000000,0,0.000000,6,6.783558,6.783558
2,242,8,218,19,0.087156,2.490881,1,0.125000,17,7.075612,7.578613
3,271,1570,70292,7833,0.111435,3.184771,364,0.231847,231962,8.033117,31.031915
4,402,4,22,0,0.000000,0.000000,0,0.000000,6,7.898527,7.898527


In [80]:
ring_df.shape

(46, 19)

In [79]:
ring_df['temporal_component'].describe()

count    46.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: temporal_component, dtype: float64

In [ ]:
G4